# Trexquant — Earnings Return Prediction (Kaggle submission)

## Input
Attach **Earnings Return Prediction Challenge 2025Q4** in the notebook **Input** panel.

## Strategy (A_raw_features parity)
- **Walk-forward CV on `di`** with warmup + **embargo** (train only on past `di`).
- Features are exactly **raw meta + raw anonymous (`f_*`)**.
- No interaction mining, no within-`di` ranks, no z-score transforms.
- **OOF Pearson** is computed on **covered** validation rows only (warmup rows excluded), not zero-filled.
- **CatBoost**; test prediction averaged over folds.

## Notes
- This notebook is intended to match `advanced_experiments.py` experiment **A_raw_features** as closely as possible.

In [ ]:
import os
import gc
import json
import warnings
from glob import glob

import numpy as np
import pandas as pd
# Optional sklearn pieces (unused by default pipeline; handy for quick experiments)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 1. Load Dataset

In [ ]:
def find_train_test():
    pairs = []
    for train_path in glob('/kaggle/input/**/train.csv', recursive=True):
        test_path = os.path.join(os.path.dirname(train_path), 'test.csv')
        if os.path.exists(test_path):
            pairs.append((train_path, test_path))
    if not pairs:
        raise FileNotFoundError(
            'No train.csv/test.csv found. Attach the competition dataset in the Kaggle Input pane first.'
        )
    return pairs[0]

train_path, test_path = find_train_test()
df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

print('train shape:', df_train.shape)
print('test  shape:', df_test.shape)

## 2. Quick audit

- Train ~146k rows, test ~72k; many `f_*` columns have heavy missingness — CatBoost handles this.
- Use **walk-forward** splits (not random KFold) so validation `di` is always after training `di`.

In [ ]:
ID_COL     = 'id'
TARGET_COL = 'target'
TIME_COL   = 'di'

feature_cols = [c for c in df_train.columns if c not in [ID_COL, TARGET_COL]]

print('id unique train:', df_train[ID_COL].is_unique)
print('id unique test: ', df_test[ID_COL].is_unique)
print('di unique train:', df_train[TIME_COL].nunique())
print('si unique train:', df_train['si'].nunique())
print()
print('target describe:')
print(df_train[TARGET_COL].describe())
print('skew:', round(df_train[TARGET_COL].skew(), 4))
print('kurt:', round(df_train[TARGET_COL].kurt(), 4))

miss = df_train[feature_cols].isna().mean().sort_values(ascending=False)
print('\nTop 10 missing:')
print(miss.head(10).to_string())

## 3. Walk-forward validation (embargo)

Same logic as `baseline_experiments.walk_forward_time_splits`: train only on **past** `di` groups; drop the last **5** `di` groups before each validation block as embargo.

In [ ]:
import math

def pearson_corr(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    if y_true.size < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float('nan')
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def walk_forward_time_splits(time_series, n_splits=5, min_train_groups=252, embargo_groups=5):
    unique_di = np.sort(time_series.dropna().unique())
    U = len(unique_di)
    warmup = max(min_train_groups, math.ceil(0.30 * U))
    remaining = unique_di[warmup:]
    chunks = np.array_split(remaining, n_splits)
    idx = np.arange(len(time_series))
    tv = time_series.to_numpy()
    splits = []
    for chunk in chunks:
        if len(chunk) == 0:
            continue
        valid_start = chunk[0]
        pre = unique_di[unique_di < valid_start]
        allowed = pre[:-embargo_groups] if embargo_groups > 0 and len(pre) > embargo_groups else np.array([], dtype=pre.dtype)
        tr_idx = idx[np.isin(tv, allowed)]
        va_idx = idx[np.isin(tv, chunk)]
        if len(tr_idx) > 0 and len(va_idx) > 0:
            splits.append((tr_idx, va_idx))
    if len(splits) < 2:
        raise ValueError('Too few walk-forward folds')
    return splits


N_SPLITS = 5
EMBARGO = 5
MIN_TRAIN_GROUPS = 252
splits = walk_forward_time_splits(
    df_train[TIME_COL], n_splits=N_SPLITS,
    min_train_groups=MIN_TRAIN_GROUPS, embargo_groups=EMBARGO,
)
print(f'Walk-forward folds (embargo={EMBARGO} di groups):')
for i, (tr, va) in enumerate(splits):
    tr_di = df_train.iloc[tr][TIME_COL]
    va_di = df_train.iloc[va][TIME_COL]
    ok = tr_di.max() < va_di.min()
    print(f'  fold {i}: train di [{tr_di.min()}–{tr_di.max()}]  valid [{va_di.min()}–{va_di.max()}]  '
          f'n_train={len(tr):,} n_valid={len(va):,}  embargo_ok={ok}')

## 4. A_raw_features CatBoost (no extra engineering)

- Use only raw columns present in both train/test: meta + anonymous `f_*`.
- Keep walk-forward folds and covered-rows OOF scoring.
- Fold-average test predictions for submission.

In [ ]:
from catboost import CatBoostRegressor

META_CANDIDATES = ['si', 'di', 'industry', 'sector', 'top2000', 'top1000', 'top500']
meta_cols = [c for c in META_CANDIDATES if c in df_train.columns]
anon_cols = sorted(c for c in df_train.columns if c.startswith('f_'))

# A_raw_features parity: raw meta + raw anonymous only.
feature_cols = [c for c in (meta_cols + anon_cols) if c in df_train.columns and c in df_test.columns]
cat_cols = [c for c in meta_cols if c in feature_cols]

train_cb = df_train[feature_cols].copy()
test_cb = df_test[feature_cols].copy()
for c in cat_cols:
    train_cb[c] = train_cb[c].astype(str)
    test_cb[c] = test_cb[c].astype(str)


class PearsonEvalMetric:
    # Match advanced_experiments.py: early stopping targets Pearson directly.
    def get_final_error(self, error, weight):
        return error

    def is_max_optimal(self):
        return True

    def evaluate(self, approxes, target, weight):
        pred = np.array(approxes[0], dtype=np.float64)
        targ = np.array(target, dtype=np.float64)
        mask = np.isfinite(pred) & np.isfinite(targ)
        if mask.sum() < 2 or np.std(pred[mask]) == 0 or np.std(targ[mask]) == 0:
            return 0.0, 1
        r = float(np.corrcoef(pred[mask], targ[mask])[0, 1])
        return (r if np.isfinite(r) else 0.0), 1


y = df_train[TARGET_COL].to_numpy(np.float64)
oof_pred = np.zeros(len(df_train), dtype=np.float64)
test_pred = np.zeros(len(df_test), dtype=np.float64)
oof_covered = np.zeros(len(df_train), dtype=bool)

print(f'Using A_raw_features with {len(feature_cols)} columns (meta={len(meta_cols)}, anon={len(anon_cols)})')

for fold_id, (tr_idx, va_idx) in enumerate(splits):
    tr_di = df_train.iloc[tr_idx][TIME_COL]
    va_di = df_train.iloc[va_idx][TIME_COL]
    assert tr_di.max() < va_di.min(), 'walk-forward violated'

    model = CatBoostRegressor(
        loss_function='RMSE',
        eval_metric=PearsonEvalMetric(),
        depth=6,
        learning_rate=0.05,
        iterations=3000,
        l2_leaf_reg=3.0,
        random_seed=42 + fold_id,
        subsample=0.8,
        bootstrap_type='Bernoulli',
        od_type='Iter',
        od_wait=300,
        verbose=False,
    )
    model.fit(
        train_cb.iloc[tr_idx], y[tr_idx],
        cat_features=cat_cols if cat_cols else None,
        eval_set=(train_cb.iloc[va_idx], y[va_idx]),
        use_best_model=True,
    )

    oof_pred[va_idx] = model.predict(train_cb.iloc[va_idx])
    oof_covered[va_idx] = True
    test_pred += model.predict(test_cb) / len(splits)

    fold_corr = pearson_corr(y[va_idx], oof_pred[va_idx])
    print(f'Fold {fold_id}: Pearson={fold_corr:.6f}  best_iter={model.best_iteration_}')

    del model
    gc.collect()

cov_frac = float(oof_covered.mean())
oof_corr = pearson_corr(y[oof_covered], oof_pred[oof_covered]) if oof_covered.sum() >= 2 else float('nan')
print(f'\nOOF Pearson (covered rows only; warmup excluded): {oof_corr:.6f}  coverage_frac={cov_frac:.4f}')
print(f'Features used: {len(feature_cols)} (interactions=0)')

## 5. Submission (A_raw_features)

**Requirements:**
- Must contain `"target"` column.
- No `nan` or `inf` in `"target"`.
- At least `10%` of target values must be non-zero.

In [ ]:
# Mild post-processing: clip extreme outliers
lo, hi = np.nanpercentile(test_pred, [0.5, 99.5])
test_pred = np.clip(test_pred, lo, hi)

# Guarantee at least 10% non-zero
if (np.abs(test_pred) > 0).mean() < 0.10:
    test_pred = test_pred + 1e-9

df_submission = pd.DataFrame({
    'id':     df_test[ID_COL],
    'target': test_pred,
})

assert np.isfinite(df_submission['target']).all(), 'Non-finite predictions found'
assert (df_submission['target'].abs() > 0).mean() >= 0.10, 'Less than 10% non-zero'

filename = '/kaggle/working/submission.csv'
df_submission.to_csv(filename, index=False)

print(f'Saved: {filename}')
print(f'Rows: {len(df_submission):,}')
print(f'Non-zero fraction: {(df_submission["target"].abs() > 0).mean():.4f}')
print(f'OOF Pearson (A_raw_features walk-forward, covered rows): {oof_corr:.6f}')
print(df_submission.head())